In [14]:
import pandas as pd
import duckdb

In [15]:
df = pd.read_csv(
    filepath_or_buffer="bmw_global_sales_2018_2025.csv"
)

In [16]:
df.head()

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.5,1.0
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.5,1.0
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.5,1.0
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.5,1.0
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.5,1.0


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3072 entries, 0 to 3071
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              3072 non-null   int64  
 1   Month             3072 non-null   int64  
 2   Region            3072 non-null   object 
 3   Model             3072 non-null   object 
 4   Units_Sold        3072 non-null   int64  
 5   Avg_Price_EUR     3072 non-null   int64  
 6   Revenue_EUR       3072 non-null   int64  
 7   BEV_Share         3072 non-null   float64
 8   Premium_Share     3072 non-null   float64
 9   GDP_Growth        3072 non-null   float64
 10  Fuel_Price_Index  3072 non-null   float64
dtypes: float64(4), int64(5), object(2)
memory usage: 264.1+ KB


In [18]:
columns = [column.lower() for column in df.columns]
df.columns = columns

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3072 entries, 0 to 3071
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              3072 non-null   int64  
 1   month             3072 non-null   int64  
 2   region            3072 non-null   object 
 3   model             3072 non-null   object 
 4   units_sold        3072 non-null   int64  
 5   avg_price_eur     3072 non-null   int64  
 6   revenue_eur       3072 non-null   int64  
 7   bev_share         3072 non-null   float64
 8   premium_share     3072 non-null   float64
 9   gdp_growth        3072 non-null   float64
 10  fuel_price_index  3072 non-null   float64
dtypes: float64(4), int64(5), object(2)
memory usage: 264.1+ KB


In [21]:
df

,year,month,region,model,units_sold,avg_price_eur,revenue_eur,bev_share,premium_share,gdp_growth,fuel_price_index
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.50,1.00
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.50,1.00
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.50,1.00
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.50,1.00
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.50,1.00
...,...,...,...,...,...,...,...,...,...,...,...
3067,2025,12,RestOfWorld,X5,9281,68198,632945638,0.201,5.89,2.37,1.41
3068,2025,12,RestOfWorld,X7,12785,91839,1174161615,0.203,5.89,2.37,1.41
3069,2025,12,RestOfWorld,i4,4906,63437,311221922,0.180,5.89,2.37,1.41
3070,2025,12,RestOfWorld,iX,7871,73867,581407157,0.196,5.89,2.37,1.41


In [36]:
ids = [num for num in range(1, 3073)]
df["id"] = ids
df

,year,month,region,model,units_sold,avg_price_eur,revenue_eur,bev_share,premium_share,gdp_growth,fuel_price_index,id
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.50,1.00,1
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.50,1.00,2
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.50,1.00,3
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.50,1.00,4
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.50,1.00,5
...,...,...,...,...,...,...,...,...,...,...,...,...
3067,2025,12,RestOfWorld,X5,9281,68198,632945638,0.201,5.89,2.37,1.41,3068
3068,2025,12,RestOfWorld,X7,12785,91839,1174161615,0.203,5.89,2.37,1.41,3069
3069,2025,12,RestOfWorld,i4,4906,63437,311221922,0.180,5.89,2.37,1.41,3070
3070,2025,12,RestOfWorld,iX,7871,73867,581407157,0.196,5.89,2.37,1.41,3071


### 🚗 1. Regional Market Leaders

### 💰 3. Top 5 Models by Revenue Share

In [45]:
duckdb.query(
    """
    with total_revenue_year as
    (
        select 
            year, 
            sum(revenue_eur) as total_revenue
        from df
        group by year 
    ),

    model_revenue as 
    (
        select 
            year,
            model,
            revenue_eur
        from df 
    )
    
    select 
        try.year,
        mr.model,
        mr.revenue_eur,
        try.total_revenue,
        round((mr.revenue_eur / try.total_revenue) * 100, 2) as revenue_share_prct
    from total_revenue_year try
    left join model_revenue mr
        on try.year = mr.year
    order by revenue_share_prct desc
    limit 5
    """
)

┌───────┬─────────┬─────────────┬───────────────┬────────────────────┐
│ year  │  model  │ revenue_eur │ total_revenue │ revenue_share_prct │
│ int64 │ varchar │    int64    │    int128     │       double       │
├───────┼─────────┼─────────────┼───────────────┼────────────────────┤
│  2019 │ X7      │  1210616110 │  176552428685 │               0.69 │
│  2021 │ X7      │  1327481990 │  194292936404 │               0.68 │
│  2019 │ X7      │  1188435264 │  176552428685 │               0.67 │
│  2021 │ X7      │  1306467472 │  194292936404 │               0.67 │
│  2025 │ X7      │  1433481712 │  215605681914 │               0.66 │
└───────┴─────────┴─────────────┴───────────────┴────────────────────┘